## A first introduction to the OpenAI API

In this part, we practice calling the OpenAI API and see how we can use its output in our scripts. We start with some basic (language-based) applications, but our goal is to use the Python REPL tool in combination with LangChain for automated code writing and testing.

In [ ]:
# Our secret key that gives us acces to OpenAI
# Be careful not to lose or share this 

from openai import OpenAI

api_key = "" 
client = OpenAI(api_key=api_key)

### Part 1: Use the ChatGPT output in Python

#### 1 Example: Changing output style of ChatGPT

We can use different personalities as the system message. Test and write your own style as well to see how ChatGPT responds to your question.

In [ ]:
# Personalities 

cynical = "You respond with skepticism, dry humor, and a pessimistic view of situations."
optimistic = "You respond with positivity, encouragement, and a hopeful outlook on every situation."
gen_z = "You respond in a casual, trendy tone using Gen Z slang, humor, and expressive language."
# Create your own style

In [ ]:
response = client.responses.create(
    model="gpt-5.4-mini-2026-03-17",
    input=[
    {"role": "system", "content": gen_z},
    {"role": "user", "content": "How should AI be used in schools?"}
]
)
print(response.output[0].content[0].text)

#### 2 Example: Working with open text fields, classification of reviews

Below you see a couple of reviews from customers for a fake restaurant. With AI, we can easily classify these reviews and provide scores on how severe these complaints are.

In [ ]:
reviews = [
    "The restaurant had a cozy atmosphere with beautiful lighting. The food was absolutely delicious, especially the pasta. Service was friendly but a bit slow.",
    "Terrible experience. The place was noisy and dirty, the food tasted bland, and the staff was rude and inattentive.",
    "Decent spot overall. The ambience was nice but nothing special. Food was good, not amazing. Service was quick and polite."
]

In [ ]:
system_prompt = """
You are a strict sentiment analysis system.

Analyze the given restaurant review and return ONLY a JSON object with the following structure:

{
  "Ambience": <score from 1 to 10>,
  "Food": <score from 1 to 10>,
  "Service": <score from 1 to 10>,
  "General": <score from 1 to 10>
}

Rules:
- Output ONLY valid JSON (no text, no explanation, no markdown)
- Scores must be integers between 1 and 10
- Base scores strictly on the content of the review
"""

In [ ]:
# Step 3: Loop through reviews and analyze
for i, review in enumerate(reviews):
    response = client.responses.create(
        model="gpt-5.4-mini-2026-03-17",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review}
        ]
    )

    output_text = response.output[0].content[0].text

    print(f"Review {i+1}:")
    print("Raw output:", output_text)

#### 3 Exercise: Working with open text fields, classification of emails

Try to write your own instructions to classify emails by priority.

In [ ]:
emails = [
    "Hi, just checking in about the meeting next week. No rush, just let me know when you're available.",
    "URGENT: The production server is down and customers cannot access the platform. We need immediate assistance!",
    "Hello, I would like to request a copy of my invoice from last month. Please send it when you have time."
]

In [ ]:
system_prompt = """
...
"""

In [ ]:
# Step 3: Loop through reviews and analyze
for i, review in enumerate(reviews):
    response = client.responses.create(
        model="gpt-5.4-mini-2026-03-17",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review}
        ]
    )

    output_text = response.output[0].content[0].text

    print(f"Review {i+1}:")
    print("Raw output:", output_text)

### Part 2: Agents

Let two agent have a debate on a topic. Define the personality of agent_1 and agent_2. Next, think of a fun debate topic and see how two agents fight it out!

In [ ]:
topic = "school is good" # For example: Is technology making us smarter?

agent_1 = f"""
You are Agent 1. 
Your presonality is ...

you argue in favor of the topic: '{topic}'. Be concise.
"""

agent_2 = f"""
You are Agent 2. 
Your presonality is ...

you argue against the topic: '{topic}'. Be concise.
"""

In [ ]:
# Create a conversation (chat) so that both agents know the past arguments

conversation = client.conversations.create()
conv_id = conversation.id 

In [ ]:
turns = 4

conversation = client.conversations.create()
conv_id = conversation.id

for i in range(turns):
    if i % 2 == 0:
        agent = "A"
        system = agent_1
        prompt = "Agent 1, respond to agent b, or if you are first, make your argument."
    else:
        agent = "B"
        system = agent_2
        prompt = "Agent B, respond to Agent A and make your counter-argument."

    response = client.responses.create(
        model="gpt-4o-mini",
        instructions=system,
        input=[{"role": "user", "content": prompt}],
        conversation=conv_id,  # persistent conversation ID
    )

    print(f"Agent {agent}: {response.output_text}\n")

#### Exercise: Pubquiz 
Use the OpenAI API to make your own pub quiz
- Use a conversation and a loop like the previous exercise (to prevent duplicate questions)
- Think of a subject for the pub quiz and proper instructions for the agent that creates the questions

Extend this script so that you can play the pub quiz in VS Code
- In the loop, both the question and your answer need to be sent to the OpenAI API, which it needs to verify
- Keep track of a score and make the questions gradually more difficult

In [ ]:
conversation = client.conversations.create()
conv_id = conversation.id 